In [1]:
import torch
import torch.nn as nn
import numpy as np

# Use double precision for stability
torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BlackScholesPINN(nn.Module):
    def __init__(self):
        super(BlackScholesPINN, self).__init__()
        # 3 inputs (S, t, K), 1 output (V)
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(), 
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

def pde_residual(model, S, t, K, sigma, r):
    """Calculates the Black-Scholes PDE residual with 3 inputs."""
    S.requires_grad_(True)
    t.requires_grad_(True)
    # K is also an input now, but we don't need its grad for the PDE
    
    # PASS ALL THREE: (S, t, K)
    V = model(torch.cat([S, t, K], dim=1))
    
    dV_dS = torch.autograd.grad(V, S, grad_outputs=torch.ones_like(V), create_graph=True)[0]
    dV_dt = torch.autograd.grad(V, t, grad_outputs=torch.ones_like(V), create_graph=True)[0]
    d2V_dS2 = torch.autograd.grad(dV_dS, S, grad_outputs=torch.ones_like(dV_dS), create_graph=True)[0]
    
    residual = dV_dt + 0.5 * (sigma**2) * (S**2) * d2V_dS2 + r * S * dV_dS - r * V
    return residual

# --- Training Setup ---
model = BlackScholesPINN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Hyperparameters (Matching our AAPL data)
r = 0.04     
sigma = 0.14  
T = 1.0      

# First, We Generate Collocation Points (or Interior)
N_pde = 10000
# Then, We Sample around the AAPL price (~270)
S_pde = (torch.rand(N_pde, 1, device=device) * 200) + 170 # Range: 170-370
t_pde = torch.rand(N_pde, 1, device=device) * T
# Variable Strike Price training
K_pde = (torch.rand(N_pde, 1, device=device) * 100) + 220 # Range: 220-320

# For Boundary Condition Points 
N_bc = 2000
S_bc = (torch.rand(N_bc, 1, device=device) * 200) + 170
t_bc = torch.full((N_bc, 1), T, device=device)
K_bc = (torch.rand(N_bc, 1, device=device) * 100) + 220 
V_bc_target = torch.clamp(S_bc - K_bc, min=0) 

# --- Training Loop ---
for epoch in range(10001):
    optimizer.zero_grad()
    
    # PDE Loss - Now passing K_pde 
    res = pde_residual(model, S_pde, t_pde, K_pde, sigma, r)
    loss_pde = torch.mean(res**2)
    
    # Boundary Loss - Now passing K_bc
    V_bc_pred = model(torch.cat([S_bc, t_bc, K_bc], dim=1))
    loss_bc = torch.mean((V_bc_pred - V_bc_target)**2)
    
    total_loss = loss_pde + loss_bc
    total_loss.backward()
    optimizer.step()
    
    if epoch % 1000 == 0:
        print(f"Epoch {epoch}: PDE Loss = {loss_pde.item():.6f}, BC Loss = {loss_bc.item():.6f}")

/scratch/mitchell/venvs/cs66-s26/lib/python3.12/site-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 0: PDE Loss = 0.500924, BC Loss = 2114.164868
Epoch 1000: PDE Loss = 51.287656, BC Loss = 373.505295
Epoch 2000: PDE Loss = 52.156349, BC Loss = 111.563040
Epoch 3000: PDE Loss = 44.952523, BC Loss = 51.594720
Epoch 4000: PDE Loss = 25.922632, BC Loss = 25.960672
Epoch 5000: PDE Loss = 12.691104, BC Loss = 14.558043
Epoch 6000: PDE Loss = 7.121376, BC Loss = 9.661650
Epoch 7000: PDE Loss = 7.984744, BC Loss = 9.159742
Epoch 8000: PDE Loss = 4.530580, BC Loss = 7.148353
Epoch 9000: PDE Loss = 3.648216, BC Loss = 6.427867
Epoch 10000: PDE Loss = 3.337303, BC Loss = 5.617520


In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime

# Setting up Ticker
ticker = yf.Ticker("AAPL")
S_current = ticker.history(period="1d")['Close'].iloc[-1]

# Getting all available expiration dates
all_expiries = ticker.options
print(f"Current Stock Price: ${S_current:.2f}")

# Pulling several expiries to get a "Time-Surface"
# We'll grab one near-term, one mid-term, and one long-term expiry
target_expiries = [all_expiries[2], all_expiries[5], all_expiries[10]] 

all_chains = []

for date in target_expiries:
    chain = ticker.option_chain(date).calls
    # Calculating Time to Maturity (t) for this specific expiry
    expiry_dt = datetime.strptime(date, '%Y-%m-%d')
    t_years = (expiry_dt - datetime.now()).days / 365.0
    
    # Cleaning the data
    temp_df = chain[['strike', 'lastPrice', 'impliedVolatility']].copy()
    temp_df['t'] = t_years
    temp_df['S'] = S_current
    temp_df['V_market'] = temp_df['lastPrice']
    
    all_chains.append(temp_df)

# Combining into one big dataset for our research
final_df = pd.concat(all_chains)

# Keeping strikes near the money (like +/- 20% of stock price)
final_df = final_df[(final_df['strike'] > S_current * 0.8) & (final_df['strike'] < S_current * 1.2)]

print(f"\nCollected {len(final_df)} data points across {len(target_expiries)} different time horizons.")
print(final_df) # Show 5 random rows

In [ ]:
# To match the market data we just pulled
K = 270.0    
S_min = 200.0
S_max = 350.0

# Updating our sampling to be in the "Apple Range"
S_pde = (torch.rand(N_pde, 1, device=device) * (S_max - S_min)) + S_min
t_pde = torch.rand(N_pde, 1, device=device) * 1.0 # 0 to 1 year

In [ ]:
S_input = torch.tensor(final_df['S'].values).unsqueeze(1).to(device)
t_input = torch.tensor(final_df['t'].values).unsqueeze(1).to(device)
K_input = torch.tensor(final_df['strike'].values).unsqueeze(1).to(device) 

# predicting
model.eval()
with torch.no_grad():
    inputs = torch.cat([S_input, t_input, K_input], dim=1)
    V_pinn = model(inputs).cpu().numpy().flatten()

# adding to df
final_df['V_pinn'] = V_pinn
final_df['Error'] = abs(final_df['V_pinn'] - final_df['V_market'])

print("Analysis Complete! Curves should now be present.")

In [ ]:
import matplotlib.pyplot as plt

times = final_df['t'].unique()

plt.figure(figsize=(10, 6))

for time_val in times:
    subset = final_df[final_df['t'] == time_val].sort_values('strike')
    plt.scatter(subset['strike'], subset['V_market'], label=f'Market (t={time_val:.2f})', alpha=0.5)
    plt.plot(subset['strike'], subset['V_pinn'], label=f'PINN (t={time_val:.2f})', linestyle='--')

plt.xlabel('Strike Price (K)')
plt.ylabel('Option Price (V)')
plt.title('PINN vs Market: Apple Option Surfaces')
plt.legend()
plt.grid(True)
plt.savefig('pinn_vs_market.png') # Put this in your paper!
plt.show()

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm

# for matplotlib config 


plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "figure.dpi": 150,
})

# black-scholes baseline
def bs_call(S, K, t, r, sigma):
    t = np.maximum(t, 1e-8)   
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * t) / (sigma * np.sqrt(t))
    d2 = d1 - sigma * np.sqrt(t)
    return S * norm.cdf(d1) - K * np.exp(-r * t) * norm.cdf(d2)


# adding analytical BS prices to final_df
final_df["V_bs"] = bs_call(
    final_df["S"].values,
    final_df["strike"].values,
    final_df["t"].values,
    r, sigma
)
final_df["Error_bs"]   = (final_df["V_bs"]   - final_df["V_market"]).abs()
final_df["Error_pinn"] = (final_df["V_pinn"] - final_df["V_market"]).abs()


final_df["moneyness"] = final_df["S"] / final_df["strike"]

# Fig 1 — PINN vs Market vs Black-Scholes, one panel per tenor

times  = sorted(final_df["t"].unique())
labels = {times[0]: "Near-term", times[1]: "Mid-term", times[2]: "Longer-dated"}
colors = {"market": "#333333", "pinn": "#E63946", "bs": "#457B9D"}

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)

for ax, t_val in zip(axes, times):
    sub = final_df[final_df["t"] == t_val].sort_values("strike")

    ax.scatter(sub["strike"], sub["V_market"],
               color=colors["market"], s=30, zorder=5,
               label="Market price")
    ax.plot(sub["strike"], sub["V_pinn"],
            color=colors["pinn"], linewidth=2, linestyle="--",
            label="PINN")
    ax.plot(sub["strike"], sub["V_bs"],
            color=colors["bs"], linewidth=1.5, linestyle=":",
            label="BSM analytical")

    ax.axvline(S_current, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.text(S_current + 1, ax.get_ylim()[0], "ATM", fontsize=7, color="gray")

    ax.set_title(f"{labels[t_val]} ($t={t_val:.2f}$yr)")
    ax.set_xlabel("Strike Price $K$ (\\$)")
    ax.set_ylabel("Option Price $V$ (\\$)")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle("PINN vs.\ BSM Analytical vs.\ Market — AAPL Call Options", fontsize=12)
plt.tight_layout()
plt.savefig("fig1_surface_comparison.png", bbox_inches="tight")
plt.show()
print("Saved: fig1_surface_comparison.png")


# Fig 2 — Absolute error by tenor: PINN vs BSM

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

for ax, t_val in zip(axes, times):
    sub = final_df[final_df["t"] == t_val].sort_values("strike")

    ax.plot(sub["strike"], sub["Error_pinn"],
            color=colors["pinn"], linewidth=2, linestyle="--", label="PINN error")
    ax.plot(sub["strike"], sub["Error_bs"],
            color=colors["bs"], linewidth=1.5, linestyle=":", label="BSM error")

    ax.set_title(f"{labels[t_val]} ($t={t_val:.2f}$yr)")
    ax.set_xlabel("Strike Price $K$ (\\$)")
    ax.set_ylabel("Absolute Error (\\$)")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle("Absolute Pricing Error: PINN vs.\ BSM Analytical", fontsize=12)
plt.tight_layout()
plt.savefig("fig2_error_by_tenor.png", bbox_inches="tight")
plt.show()
print("Saved: fig2_error_by_tenor.png")


# fig 3 - Volatility Smile: implied vol from market vs constant sigma

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

for ax, t_val in zip(axes, times):
    sub = final_df[final_df["t"] == t_val].sort_values("strike")

    ax.scatter(sub["strike"], sub["impliedVolatility"],
               color=colors["market"], s=25, zorder=5,
               label="Market implied vol")
    ax.axhline(sigma, color=colors["bs"], linewidth=1.5, linestyle="--",
               label=f"Constant $\\sigma={sigma}$")

    ax.set_title(f"{labels[t_val]} ($t={t_val:.2f}$yr)")
    ax.set_xlabel("Strike Price $K$ (\\$)")
    ax.set_ylabel("Implied Volatility")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle("Volatility Smile: Market Implied Vol vs.\ BSM Constant $\\sigma$", fontsize=12)
plt.tight_layout()
plt.savefig("fig3_vol_smile.png", bbox_inches="tight")
plt.show()
print("Saved: fig3_vol_smile.png")


# model surface
model.eval()

S_grid = torch.linspace(200, 350, 80, device=device, dtype=torch.float64)
K_fixed = torch.tensor(270.0, device=device, dtype=torch.float64)
t_vals_plot = torch.tensor(times, device=device, dtype=torch.float64)

fig, ax = plt.subplots(figsize=(7, 4))

for t_val in t_vals_plot:
    S_in = S_grid.unsqueeze(1).requires_grad_(True)
    t_in = torch.full((len(S_grid), 1), t_val.item(), device=device,
                      dtype=torch.float64)
    K_in = torch.full((len(S_grid), 1), K_fixed.item(), device=device,
                      dtype=torch.float64)

    V = model(torch.cat([S_in, t_in, K_in], dim=1))
    delta = torch.autograd.grad(V, S_in,
                                grad_outputs=torch.ones_like(V),
                                create_graph=False)[0]

    ax.plot(S_grid.cpu().detach().numpy(),
            delta.cpu().detach().numpy().flatten(),
            label=f"$t={t_val.item():.2f}$yr")

ax.axvline(K_fixed.item(), color="gray", linestyle="--",
           linewidth=0.8, alpha=0.6, label=f"Strike $K={K_fixed.item():.0f}$")
ax.set_xlabel("Stock Price $S$ (\\$)")
ax.set_ylabel("Delta $\\Delta = \\partial V / \\partial S$")
ax.set_title("PINN-Derived Delta Surface ($K=270$)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig4_delta_surface.png", bbox_inches="tight")
plt.show()
print("Saved: fig4_delta_surface.png")


# table 1 — MAE and relative error by tenor, PINN vs BSM
rows = []
for t_val in times:
    sub = final_df[final_df["t"] == t_val]
    n   = len(sub)
    mae_pinn = sub["Error_pinn"].mean()
    mae_bs   = sub["Error_bs"].mean()
    rel_pinn = (sub["Error_pinn"] / sub["V_market"]).mean() * 100
    rel_bs   = (sub["Error_bs"]  / sub["V_market"]).mean() * 100
    rows.append({
        "Tenor (yr)": f"{t_val:.2f}",
        "N": n,
        "MAE PINN ($)": f"{mae_pinn:.3f}",
        "MAE BSM ($)":  f"{mae_bs:.3f}",
        "Rel. Err. PINN (%)": f"{rel_pinn:.1f}",
        "Rel. Err. BSM (%)":  f"{rel_bs:.1f}",
    })

# Overall row
rows.append({
    "Tenor (yr)": "Overall",
    "N": len(final_df),
    "MAE PINN ($)": f"{final_df['Error_pinn'].mean():.3f}",
    "MAE BSM ($)":  f"{final_df['Error_bs'].mean():.3f}",
    "Rel. Err. PINN (%)": f"{(final_df['Error_pinn']/final_df['V_market']).mean()*100:.1f}",
    "Rel. Err. BSM (%)":  f"{(final_df['Error_bs'] /final_df['V_market']).mean()*100:.1f}",
})

results_df = pd.DataFrame(rows)
print("\n=== TABLE 1: Pricing Error by Tenor ===")
print(results_df.to_string(index=False))

print("\n=== LaTeX version ===")
print(results_df.to_latex(index=False, escape=False))